In [1]:
from __future__ import annotations

import os

from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import hashlib

from torch_geometric.data import Data, Dataset as PyGDataset

from qqe.GNN.physics_aware_NN import GNN
import torch.nn as nn
from torch.optim import Adam
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import random_split
from torch_geometric.loader import DataLoader

from qqe.GNN.physics_aware_NN import GNN, QuantumCircuitGraphDataset

In [2]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

2.5.1+cu121
12.1
True
Using device: cuda


In [3]:
MODEL_STATE_PATH = "../models/gnn_model.pt"
CHECKPOINT_PATH = "../models/gnn_checkpoint.pt"


In [4]:
def collect_pt_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    # support either dataset_dir/*.pt or dataset_dir/samples/*.pt
    if family is not None:
        paths = sorted((d / "encoding_data_pennylane" / family).glob("*.pt"))
    else:
        paths = []
        encoding_dir = d / "encoding_data_pennylane"
        if encoding_dir.exists():
            for family_dir in sorted(encoding_dir.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))
    if not paths:
        paths = sorted(d.glob("*.pt"))
    return [str(p) for p in paths]

def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    canonical = "|".join(sorted(Path(p).name for p in paths))

    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]

    tag = f"_{suffix}" if suffix else ""

    cache_dir = Path("qqe") / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)

    return str(cache_dir)

In [5]:
class PaddedGraphDatasetWrapper:
    """Wrapper that pads/truncates graph features to consistent dimensions."""

    def __init__(
        self,
        dataset,
        target_node_dim: int | None = None,
        target_global_dim: int | None = None,
        target_dim: int | None = None,  # backwards compatibility
    ):
        self.dataset = dataset

        # Backwards compatibility with older call sites using target_dim.
        if target_node_dim is None and target_dim is not None:
            target_node_dim = target_dim

        self.target_dim = target_node_dim if target_node_dim is not None else self._compute_max_node_dim()
        self.target_global_dim = (
            target_global_dim if target_global_dim is not None else self._compute_max_global_dim()
        )

    def _compute_max_node_dim(self) -> int:
        """Find max node feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            x = getattr(data, "x", None)
            if x is not None and x.dim() > 1:
                max_dim = max(max_dim, int(x.shape[1]))
        return max_dim

    def _compute_max_global_dim(self) -> int:
        """Find max global feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            g = getattr(data, "global_features", None)
            if g is None:
                continue
            if g.dim() == 0:
                width = 1
            elif g.dim() == 1:
                width = int(g.shape[0])
            else:
                width = int(g.shape[-1])
            max_dim = max(max_dim, width)
        return max_dim

    def _fit_node_dim(self, data):
        x = getattr(data, "x", None)
        if x is None or x.dim() <= 1:
            return data
        current = int(x.shape[1])
        if current == self.target_dim:
            return data
        out = data.clone()
        if current < self.target_dim:
            pad_size = self.target_dim - current
            out.x = torch.nn.functional.pad(out.x, (0, pad_size), mode="constant", value=0.0)
        else:
            out.x = out.x[:, : self.target_dim]
        return out

    def _fit_global_dim(self, data):
        g = getattr(data, "global_features", None)
        if g is None or self.target_global_dim <= 0:
            return data

        out = data.clone() if out_is_same(data, g) else data
        g = getattr(out, "global_features")

        # Ensure graph-level vector shape.
        if g.dim() == 0:
            g = g.view(1)
        elif g.dim() > 1:
            g = g.view(-1)

        current = int(g.shape[0])
        if current < self.target_global_dim:
            g = torch.nn.functional.pad(
                g, (0, self.target_global_dim - current), mode="constant", value=0.0
            )
        elif current > self.target_global_dim:
            g = g[: self.target_global_dim]

        out.global_features = g
        return out

    def __getitem__(self, idx: int):
        data = self.dataset[idx]
        data = self._fit_node_dim(data)
        data = self._fit_global_dim(data)
        return data

    def __len__(self) -> int:
        return len(self.dataset)


def out_is_same(data, g):
    # Clone lazily only when we actually need to edit global features.
    return hasattr(data, "global_features") and data.global_features is g

In [6]:
def build_train_val_test_loaders_two_stage(
    pt_paths: list[str],
    train_split: float = 0.8,
    val_within_train: float = 0.1,
    batch_size: int = 32,
    seed: int = 42,
    global_feature_variant: str = "baseline",
    node_feature_backend_variant: str | None = None,
) -> tuple[QuantumCircuitGraphDataset, PaddedGraphDatasetWrapper,DataLoader, DataLoader, DataLoader, int, int]:
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )

    if len(dataset) < 3:
        raise RuntimeError("Dataset too small for train/val/test splitting.")

    padded_dataset = PaddedGraphDatasetWrapper(dataset)
    node_in_dim = padded_dataset.target_dim
    global_in_dim = dataset.global_feature_dim

    generator = torch.Generator().manual_seed(seed)
    primary_train_len = max(1, int(len(padded_dataset) * train_split))
    test_len = max(1, len(padded_dataset) - primary_train_len)
    while primary_train_len + test_len > len(padded_dataset):
        primary_train_len -= 1

    primary_train, test_ds = random_split(
        padded_dataset,
        [primary_train_len, test_len],
        generator=generator,
    )

    val_len = max(1, int(len(primary_train) * val_within_train))
    real_train_len = max(1, len(primary_train) - val_len)
    train_ds, val_ds = random_split(
        primary_train,
        [real_train_len, val_len],
        generator=generator,
    )

    pin_mem = torch.cuda.is_available()

    return (
        dataset,
        padded_dataset,
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_mem),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_mem),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_mem),
        node_in_dim,
        global_in_dim,
    )

In [7]:
family = "quansistor"

In [8]:
data_paths = collect_pt_paths("../outputs/data", family=family)

In [9]:
print(f"Collected {len(data_paths)} .pt files for dataset.")

Collected 0 .pt files for dataset.


In [10]:
for path in data_paths[:5]:  # Print first 5 paths as a sanity check
    print(path)

In [11]:
dataset, padded_data, train_loader, val_loader, test_loader, node_in_dim, global_in_dim = build_train_val_test_loaders_two_stage(
    data_paths,  # Use first 10 paths for demonstration
    train_split=0.8,
    val_within_train=0.1,
    batch_size=32,
    seed=42,
    global_feature_variant="binned",
    node_feature_backend_variant=None,
)

RuntimeError: Dataset too small for train/val/test splitting.

In [ ]:
padded_data

In [ ]:
dataset

QuantumCircuitGraphDataset(10000)

In [ ]:
for i, sample in enumerate(dataset):
    x_shape = tuple(sample.x.shape) if getattr(sample, "x", None) is not None else None
    y_val = sample.y.item() if getattr(sample, "y", None) is not None and sample.y.numel() == 1 else getattr(sample, "y", None)
    print(f"{i}: nodes={sample.num_nodes}, edges={sample.num_edges}, x_shape={x_shape}, y={y_val}")
    print(f"   global features shape: {sample.global_features.shape if hasattr(sample, 'global_features') else None}")
    # print(f"   node feature sample: {sample.x[0] if hasattr(sample, 'x') and sample.x.shape[0] > 0 else None}")

    if i >= 9:
        print("...showing first 10 samples only")
        break

0: nodes=70, edges=110, x_shape=(70, 23), y=7.637660503387451
   global features shape: torch.Size([1, 302])
1: nodes=70, edges=110, x_shape=(70, 23), y=7.809410572052002
   global features shape: torch.Size([1, 302])
2: nodes=70, edges=110, x_shape=(70, 23), y=7.66350793838501
   global features shape: torch.Size([1, 302])
3: nodes=70, edges=110, x_shape=(70, 23), y=7.81531286239624
   global features shape: torch.Size([1, 302])
4: nodes=70, edges=110, x_shape=(70, 23), y=7.843303680419922
   global features shape: torch.Size([1, 302])
5: nodes=70, edges=110, x_shape=(70, 23), y=7.698472023010254
   global features shape: torch.Size([1, 302])
6: nodes=70, edges=110, x_shape=(70, 23), y=7.704659461975098
   global features shape: torch.Size([1, 302])
7: nodes=70, edges=110, x_shape=(70, 23), y=7.437838077545166
   global features shape: torch.Size([1, 302])
8: nodes=70, edges=110, x_shape=(70, 23), y=7.5361762046813965
   global features shape: torch.Size([1, 302])
9: nodes=70, edges=1

In [ ]:
MASTER_GATE_TYPES = [
    "input", "measurement",
    "h", "s", "t", "id",
    "rx", "ry", "rz",
    "cx",
    "qx", "qy", "haar",
]

FAMILY_GATE_TYPES = {
    "random": ["input", "measurement", "rx", "ry", "rz", "cx"],
    "clifford": ["input", "measurement", "h", "s", "t", "id", "cx"],
    "haar": ["input", "measurement", "haar"],
    "quansistor": ["input", "measurement", "qx", "qy"],
}

In [ ]:
class FamilyNodeProjector:
    def __init__(self, family: str):
        self.family = family
        self.master_gate_types = MASTER_GATE_TYPES
        self.family_gate_types = FAMILY_GATE_TYPES[family]
        self.keep_gate_idx = [
            self.master_gate_types.index(name) for name in self.family_gate_types
        ]
        self.n_gate_master = len(self.master_gate_types)

    def __call__(self, data: Data) -> Data:
        gate_part = data.x[:, :self.n_gate_master]
        qubit_part = data.x[:, self.n_gate_master:]

        out = data.clone()
        out.x = torch.cat([gate_part[:, self.keep_gate_idx], qubit_part], dim=1)
        return out

In [ ]:
projector = FamilyNodeProjector(family)
for i, sample in enumerate(dataset):
    sample = projector(sample)
    x_shape = tuple(sample.x.shape) if getattr(sample, "x", None) is not None else None
    y_val = sample.y.item() if getattr(sample, "y", None) is not None and sample.y.numel() == 1 else getattr(sample, "y", None)
    print(f"{i}: nodes={sample.num_nodes}, edges={sample.num_edges}, x_shape={x_shape}, y={y_val}")
    print(f"   global features shape: {sample.global_features.shape if hasattr(sample, 'global_features') else None}")
    # print(f"   node feature sample: {sample.x[0] if hasattr(sample, 'x') and sample.x.shape[0] > 0 else None}")

    if i >= 9:
        print("...showing first 10 samples only")
        break

0: nodes=70, edges=110, x_shape=(70, 14), y=7.637660503387451
   global features shape: torch.Size([1, 302])
1: nodes=70, edges=110, x_shape=(70, 14), y=7.809410572052002
   global features shape: torch.Size([1, 302])
2: nodes=70, edges=110, x_shape=(70, 14), y=7.66350793838501
   global features shape: torch.Size([1, 302])
3: nodes=70, edges=110, x_shape=(70, 14), y=7.81531286239624
   global features shape: torch.Size([1, 302])
4: nodes=70, edges=110, x_shape=(70, 14), y=7.843303680419922
   global features shape: torch.Size([1, 302])
5: nodes=70, edges=110, x_shape=(70, 14), y=7.698472023010254
   global features shape: torch.Size([1, 302])
6: nodes=70, edges=110, x_shape=(70, 14), y=7.704659461975098
   global features shape: torch.Size([1, 302])
7: nodes=70, edges=110, x_shape=(70, 14), y=7.437838077545166
   global features shape: torch.Size([1, 302])
8: nodes=70, edges=110, x_shape=(70, 14), y=7.5361762046813965
   global features shape: torch.Size([1, 302])
9: nodes=70, edges=1

In [ ]:
sample

Data(
  x=[70, 14],
  edge_index=[2, 110],
  y=[1],
  global_features=[1, 302],
  num_qubits=10,
  gate_counts={
    qx_a_bin_0=0,
    qx_a_bin_1=2,
    qx_a_bin_10=0,
    qx_a_bin_11=0,
    qx_a_bin_12=0,
    qx_a_bin_13=0,
    qx_a_bin_14=0,
    qx_a_bin_15=0,
    qx_a_bin_16=1,
    qx_a_bin_17=1,
    qx_a_bin_18=0,
    qx_a_bin_19=0,
    qx_a_bin_2=1,
    qx_a_bin_20=2,
    qx_a_bin_21=1,
    qx_a_bin_22=1,
    qx_a_bin_23=0,
    qx_a_bin_24=0,
    qx_a_bin_25=0,
    qx_a_bin_26=0,
    qx_a_bin_27=2,
    qx_a_bin_28=1,
    qx_a_bin_29=0,
    qx_a_bin_3=1,
    qx_a_bin_30=0,
    qx_a_bin_31=2,
    qx_a_bin_32=0,
    qx_a_bin_33=0,
    qx_a_bin_34=0,
    qx_a_bin_35=0,
    qx_a_bin_36=1,
    qx_a_bin_37=0,
    qx_a_bin_38=1,
    qx_a_bin_39=1,
    qx_a_bin_4=0,
    qx_a_bin_40=0,
    qx_a_bin_41=0,
    qx_a_bin_42=1,
    qx_a_bin_43=1,
    qx_a_bin_44=0,
    qx_a_bin_45=0,
    qx_a_bin_46=0,
    qx_a_bin_47=2,
    qx_a_bin_48=0,
    qx_a_bin_49=0,
    qx_a_bin_5=1,
    qx_a_bin_6=1,
 

In [ ]:
def collect_pred_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    pred_root = d / "predictions"

    if family is not None:
        paths = sorted((pred_root / family).glob("*.pt"))
    else:
        paths = []
        if pred_root.exists():
            for family_dir in sorted(pred_root.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))

    if not paths:
        paths = sorted(d.glob("*.pt"))

    return [str(p.resolve()) for p in paths]


def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    # Include full resolved paths in the digest to avoid collisions across folders/families.
    canonical = "|".join(sorted(str(Path(p).resolve()) for p in paths))
    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]
    tag = f"_{suffix}" if suffix else ""
    cache_dir = Path("..") / "qqe" / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)
    return str(cache_dir.resolve())

In [ ]:
family = "clifford"
global_feature_variant = "binned"
node_feature_backend_variant = None

In [ ]:
pred_data_paths = collect_pred_paths("../outputs/data", family=family)

In [ ]:
print(f"Collected {len(pred_data_paths)} .pt files for dataset.")

Collected 8750 .pt files for dataset.


In [ ]:
def build_pred_loaders_two_stage(
    pt_paths: list[str],
    batch_size: int = 32,
    seed: int = 42,
    global_feature_variant: str = "baseline",
    node_feature_backend_variant: str | None = None,
) -> tuple[QuantumCircuitGraphDataset, PaddedGraphDatasetWrapper,DataLoader, int, int]:
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )

    if len(dataset) < 3:
        raise RuntimeError("Dataset too small for train/val/test splitting.")

    padded_dataset = PaddedGraphDatasetWrapper(dataset)
    node_in_dim = padded_dataset.target_dim
    global_in_dim = dataset.global_feature_dim

    pred_ds = padded_dataset
    pin_mem = torch.cuda.is_available()
    return (
        dataset,
        padded_dataset,
        DataLoader(pred_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_mem),
        node_in_dim,
        global_in_dim,
    )

In [ ]:
dataset, padded_data, pred_loader, node_in_dim, global_in_dim = build_pred_loaders_two_stage(
    pred_data_paths,  # Use first 10 paths for demonstration
    batch_size=32,
)

C:\Users\Victor\Desktop\Université\Research\qml-quansistor-entropy\qqe\GNN\physics_aware_NN.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  obj = torch.load(pt_path, m

In [ ]:
for i, sample in enumerate(dataset):
    x_shape = tuple(sample.x.shape) if getattr(sample, "x", None) is not None else None
    print(f"{i}: nodes={sample.num_nodes}, edges={sample.num_edges}, x_shape={x_shape}")
    print(f"   global features shape: {sample.global_features.shape if hasattr(sample, 'global_features') else None}")
    # print(f"   node feature sample: {sample.x[0] if hasattr(sample, 'x') and sample.x.shape[0] > 0 else None}")

    if i >= 9:
        print("...showing first 10 samples only")
        break

0: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
1: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
2: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
3: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
4: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
5: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
6: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
7: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
8: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
9: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
...showing first 10 samples only


In [ ]:
sample = dataset[0]
print(sample)

Data(
  x=[227, 25],
  edge_index=[2, 276],
  y=[1],
  global_features=[1, 7],
  num_qubits=12,
  gate_counts={
    CNOT_count=61,
    H_count=32,
    I_count=34,
    S_count=56,
    T_count=20,
  },
  meta={
    cid='clifford_Q12_L11_S1204654993',
    family='clifford',
    n_qubits=12,
    n_layers=11,
    seed=1204654993,
    backend='pennylane',
    method='fwht',
    representation='dense',
    n_bins=50,
  }
)


In [ ]:
# --------------------------------------------------
# 1. Build prediction dataset with training-compatible feature config
# --------------------------------------------------
if not pred_data_paths:
    raise RuntimeError("No prediction .pt files found. Check outputs/data/predictions/<family>.")

checkpoint = None
model_config = {}
fixed_all_gate_keys = None

if Path(CHECKPOINT_PATH).exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    if isinstance(checkpoint, dict):
        model_config = checkpoint.get("model_config", {}) or {}
        feature_config = checkpoint.get("feature_config", {}) or {}
        fixed_all_gate_keys = feature_config.get("all_gate_keys", None)

suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
root = _cache_root_for_paths(pred_data_paths, suffix=suffix)

pred_dataset = QuantumCircuitGraphDataset(
    root=root,
    pt_paths=pred_data_paths,
    global_feature_variant=global_feature_variant,
    node_feature_backend_variant=node_feature_backend_variant,
    fixed_all_gate_keys=fixed_all_gate_keys,
)

trained_node_in_dim = model_config.get("node_in_dim", None)
trained_global_in_dim = model_config.get("global_in_dim", None)

padded_pred_dataset = PaddedGraphDatasetWrapper(
    pred_dataset,
    target_node_dim=trained_node_in_dim if trained_node_in_dim is not None else None,
    target_global_dim=trained_global_in_dim if trained_global_in_dim is not None else None,
    target_dim=trained_node_in_dim if trained_node_in_dim is not None else None,
 )

pred_loader = DataLoader(
    padded_pred_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

node_in_dim = padded_pred_dataset.target_dim
global_in_dim = padded_pred_dataset.target_global_dim

print(f"Prediction graphs: {len(pred_dataset)}")
print(f"Prediction node feature dim: {node_in_dim}")
print(f"Prediction global feature dim: {global_in_dim}")
if trained_node_in_dim is not None or trained_global_in_dim is not None:
    print(f"Trained node/global dims from checkpoint: {trained_node_in_dim}/{trained_global_in_dim}")

C:\Users\Victor\AppData\Local\Temp\ipykernel_152404\351848649.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu

Prediction graphs: 8750
Prediction node feature dim: 23
Prediction global feature dim: 458
Trained node/global dims from checkpoint: 23/458


In [ ]:
print(pred_data_paths[:5])

['C:\\Users\\Victor\\Desktop\\Université\\Research\\qml-quansistor-entropy\\outputs\\data\\predictions\\clifford\\clifford_Q12_L11_S1204654993.pt', 'C:\\Users\\Victor\\Desktop\\Université\\Research\\qml-quansistor-entropy\\outputs\\data\\predictions\\clifford\\clifford_Q12_L11_S125238218.pt', 'C:\\Users\\Victor\\Desktop\\Université\\Research\\qml-quansistor-entropy\\outputs\\data\\predictions\\clifford\\clifford_Q12_L11_S1306310596.pt', 'C:\\Users\\Victor\\Desktop\\Université\\Research\\qml-quansistor-entropy\\outputs\\data\\predictions\\clifford\\clifford_Q12_L11_S1507634226.pt', 'C:\\Users\\Victor\\Desktop\\Université\\Research\\qml-quansistor-entropy\\outputs\\data\\predictions\\clifford\\clifford_Q12_L11_S1592511977.pt']


In [ ]:
pred_dataset

QuantumCircuitGraphDataset(8750)

In [ ]:
# --------------------------------------------------
# 2. Validate dataset/model dimensions
# --------------------------------------------------
print("node_in_dim (prediction) =", node_in_dim)
print("global_in_dim (prediction) =", global_in_dim)
print("node_in_dim (trained) =", trained_node_in_dim)
print("global_in_dim (trained) =", trained_global_in_dim)

if trained_node_in_dim is not None and node_in_dim != trained_node_in_dim:
    print(f"Node dim will be adapted to trained dim {trained_node_in_dim} at inference time.")

if trained_global_in_dim is not None and global_in_dim != trained_global_in_dim:
    print(f"Global feature dim will be adapted to trained dim {trained_global_in_dim} at inference time.")

node_in_dim (prediction) = 23
global_in_dim (prediction) = 458
node_in_dim (trained) = 23
global_in_dim (trained) = 458


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# --------------------------------------------------
# 3. Rebuild model and load weights robustly
# --------------------------------------------------
def _extract_state_dict(payload):
    if isinstance(payload, nn.Module):
        return payload.state_dict()
    if isinstance(payload, dict) and "model_state_dict" in payload:
        return payload["model_state_dict"]
    if isinstance(payload, dict) and all(torch.is_tensor(v) for v in payload.values()):
        return payload
    raise RuntimeError("Unsupported model file format.")

if checkpoint is not None and isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    if not Path(MODEL_STATE_PATH).exists():
        raise FileNotFoundError(f"Could not find model weights at {MODEL_STATE_PATH}")
    raw_payload = torch.load(MODEL_STATE_PATH, map_location="cpu")
    state_dict = _extract_state_dict(raw_payload)

model_kwargs = {
    "node_in_dim": int(trained_node_in_dim if trained_node_in_dim is not None else node_in_dim),
    "global_in_dim": int(trained_global_in_dim if trained_global_in_dim is not None else global_in_dim),
    "gnn_hidden": int(model_config.get("gnn_hidden", 32)),
    "gnn_heads": int(model_config.get("gnn_heads", 8)),
    "global_hidden": int(model_config.get("global_hidden", 16)),
    "reg_hidden": int(model_config.get("reg_hidden", 16)),
    "num_layers": int(model_config.get("num_layers", 5)),
    "dropout_rate": float(model_config.get("dropout_rate", 0.1)),
}

model = GNN(**model_kwargs).to(device)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
model.eval()

print("Loaded model config:", model_kwargs)
if missing_keys:
    print("Missing keys:", missing_keys)
if unexpected_keys:
    print("Unexpected keys:", unexpected_keys)

Loaded model config: {'node_in_dim': 23, 'global_in_dim': 458, 'gnn_hidden': 32, 'gnn_heads': 8, 'global_hidden': 16, 'reg_hidden': 16, 'num_layers': 5, 'dropout_rate': 0.1}


In [ ]:
# --------------------------------------------------
# 4. Predict with defensive dimension adaptation
# --------------------------------------------------
predictions = []
expected_node_dim = model_kwargs["node_in_dim"]
expected_global_dim = model_kwargs["global_in_dim"]

with torch.no_grad():
    for batch in pred_loader:
        # Adapt node features to the model input dimension.
        if batch.x.size(1) < expected_node_dim:
            pad_size = expected_node_dim - batch.x.size(1)
            batch.x = torch.nn.functional.pad(batch.x, (0, pad_size), mode="constant", value=0.0)
        elif batch.x.size(1) > expected_node_dim:
            batch.x = batch.x[:, :expected_node_dim]

        # Adapt global features to the model input dimension.
        g = batch.global_features
        if g.dim() == 1:
            g = g.view(batch.num_graphs, -1)
        if g.size(1) < expected_global_dim:
            g = torch.nn.functional.pad(g, (0, expected_global_dim - g.size(1)), mode="constant", value=0.0)
        elif g.size(1) > expected_global_dim:
            g = g[:, :expected_global_dim]
        batch.global_features = g

        batch = batch.to(device)
        pred = model(batch).view(-1)
        predictions.extend(pred.cpu().tolist())

print("First 10 predictions:", predictions[:10])
print("Total predictions:", len(predictions))

In [ ]:
predictions

[2.678736686706543,
 2.456918716430664,
 2.477470636367798,
 2.523822546005249,
 2.448845386505127,
 2.61976957321167,
 2.527053117752075,
 2.541417121887207,
 2.555988073348999,
 2.64188551902771,
 2.4498233795166016,
 2.5020012855529785,
 2.405000686645508,
 2.532179832458496,
 2.4252524375915527,
 2.652402877807617,
 2.4280452728271484,
 2.7054128646850586,
 2.5823898315429688,
 2.627990245819092,
 2.5880889892578125,
 2.628566265106201,
 2.67242431640625,
 2.616528034210205,
 2.8187496662139893,
 3.0201447010040283,
 2.9299862384796143,
 3.147155523300171,
 3.039719343185425,
 2.922070026397705,
 2.83526873588562,
 3.0249202251434326,
 2.9404067993164062,
 2.8971199989318848,
 2.9890761375427246,
 2.979753255844116,
 2.9479148387908936,
 3.124241352081299,
 2.9788267612457275,
 2.920698642730713,
 3.0564963817596436,
 3.0449249744415283,
 2.9815051555633545,
 2.8271546363830566,
 3.013493061065674,
 3.1649415493011475,
 3.074591636657715,
 2.985894203186035,
 2.9456982612609863,
 3